In [1]:
import pandas as pd
import os
import sys

# Add project root to PYTHONPATH
project_root = os.path.abspath("../")
sys.path.append(project_root)

from src.preprocessing.load_data import load_dataset
from src.preprocessing.clean_data import clean_and_augment
from src.preprocessing.feature_engineering import add_features
import importlib
import src.preprocessing.clean_data as clean_module
importlib.reload(clean_module)

from src.preprocessing.clean_data import clean_and_augment

In [5]:
raw_path = "../data/raw/Final_data_set2.csv"   # update filename
df_raw = load_dataset(raw_path)

print("Raw Dataset Loaded:")
df_raw.head()


[INFO] Loading dataset from: ../data/raw/Final_data_set2.csv
[INFO] Standardized columns: ['timestamp', 'load_actual', 'temperature', 'humidity', 'dew_point', 'solar_generation', 'wind_generation']
[INFO] Rows: 43848
Raw Dataset Loaded:


,timestamp,load_actual,temperature,humidity,dew_point,solar_generation,wind_generation
0,2014-12-31T23:00,NaN,0.5,96.0,-0.1,NaN,NaN
1,2015-01-01T00:00,9484.0,0.1,95.0,-0.5,NaN,NaN
2,2015-01-01T01:00,9152.0,-0.2,95.0,-0.9,NaN,734.81
3,2015-01-01T02:00,8799.0,-0.4,95.0,-1.2,NaN,766.64
4,2015-01-01T03:00,8567.0,-0.6,95.0,-1.3,NaN,733.13


In [6]:
df_raw["timestamp"] = pd.to_datetime(df_raw["timestamp"], errors="coerce")
print(df_raw["timestamp"].dtype)
print(df_raw["timestamp"].isna().sum())


datetime64[ns]
23


In [7]:
print("HEAD TIMESTAMP:")
print(df_raw["timestamp"].head(10))

print("\nTIMESTAMP DTYPE:")
print(df_raw["timestamp"].dtype)

print("\nAFTER SETTING INDEX:")
df_temp = df_raw.copy()
df_temp["timestamp"] = pd.to_datetime(df_temp["timestamp"], errors="coerce")
df_temp = df_temp.set_index("timestamp")
print(df_temp.index.dtype)

print("\nCHECK FOR DUPLICATES:")
print(df_temp.index.duplicated().sum())


HEAD TIMESTAMP:
0   2014-12-31 23:00:00
1   2015-01-01 00:00:00
2   2015-01-01 01:00:00
3   2015-01-01 02:00:00
4   2015-01-01 03:00:00
5   2015-01-01 04:00:00
6   2015-01-01 05:00:00
7   2015-01-01 06:00:00
8   2015-01-01 07:00:00
9   2015-01-01 08:00:00
Name: timestamp, dtype: datetime64[ns]

TIMESTAMP DTYPE:
datetime64[ns]

AFTER SETTING INDEX:
datetime64[ns]

CHECK FOR DUPLICATES:
22


In [8]:
print(df_raw[[
    "load_actual",
    "temperature",
    "humidity",
    "dew_point",
    "solar_generation",
    "wind_generation"
]].dtypes)


load_actual         float64
temperature         float64
humidity            float64
dew_point           float64
solar_generation    float64
wind_generation     float64
dtype: object


In [9]:
for col in [
    "load_actual",
    "temperature",
    "humidity",
    "dew_point",
    "solar_generation",
    "wind_generation"
]:
    print(f"\nCHECKING COLUMN: {col}")
    print(df_raw[col].head(10))
    print("UNIQUE NON-NUMERIC VALUES:", df_raw[col].apply(lambda x: isinstance(x, str)).sum())



CHECKING COLUMN: load_actual
0       NaN
1    9484.0
2    9152.0
3    8799.0
4    8567.0
5    8487.0
6    8428.0
7    8122.0
8    8179.0
9    8340.0
Name: load_actual, dtype: float64
UNIQUE NON-NUMERIC VALUES: 0

CHECKING COLUMN: temperature
0    0.5
1    0.1
2   -0.2
3   -0.4
4   -0.6
5   -0.6
6   -0.5
7   -0.5
8    0.6
9    0.6
Name: temperature, dtype: float64
UNIQUE NON-NUMERIC VALUES: 0

CHECKING COLUMN: humidity
0    96.0
1    95.0
2    95.0
3    95.0
4    95.0
5    94.0
6    94.0
7    94.0
8    96.0
9    96.0
Name: humidity, dtype: float64
UNIQUE NON-NUMERIC VALUES: 0

CHECKING COLUMN: dew_point
0   -0.1
1   -0.5
2   -0.9
3   -1.2
4   -1.3
5   -1.4
6   -1.4
7   -1.3
8    0.1
9    0.1
Name: dew_point, dtype: float64
UNIQUE NON-NUMERIC VALUES: 0

CHECKING COLUMN: solar_generation
0      NaN
1      NaN
2      NaN
3      NaN
4      NaN
5      NaN
6      NaN
7      NaN
8      NaN
9    92.66
Name: solar_generation, dtype: float64
UNIQUE NON-NUMERIC VALUES: 0

CHECKING COLUMN: wind_ge

In [10]:
print(df_raw["timestamp"].is_monotonic_increasing)


False


In [11]:
print(df_raw["timestamp"].diff().min())


0 days 01:00:00


In [12]:
df_raw = df_raw.sort_index()


In [13]:
print("Monotonic increasing?:", df_raw["timestamp"].is_monotonic_increasing)
print("Minimum time difference:", df_raw["timestamp"].diff().min())


Monotonic increasing?: False
Minimum time difference: 0 days 01:00:00


In [14]:
df_clean = clean_and_augment(df_raw)

print("After Cleaning & Augmentation:")
df_clean.head()


[INFO] Cleaning dataset...
[INFO] Linear interpolation applied successfully.
[INFO] Creating augmented dataset...
[INFO] Final rows after augmentation: 87650
After Cleaning & Augmentation:


,timestamp,load_actual,temperature,humidity,dew_point,solar_generation,wind_generation
0,2014-12-31 23:00:00,9484.0,0.5,96.0,-0.1,92.66,734.81
1,2015-01-01 00:00:00,9484.0,0.1,95.0,-0.5,92.66,734.81
2,2015-01-01 01:00:00,9152.0,-0.2,95.0,-0.9,92.66,734.81
3,2015-01-01 02:00:00,8799.0,-0.4,95.0,-1.2,92.66,766.64
4,2015-01-01 03:00:00,8567.0,-0.6,95.0,-1.3,92.66,733.13


In [15]:
# Step: Save processed dataset
save_path = "../data/processed/final_dataset2.csv"
df_clean.to_csv(save_path, index=False)

print("[INFO] Processed file saved to:", save_path)
print("[INFO] File size:", os.path.getsize(save_path), "bytes")


[INFO] Processed file saved to: ../data/processed/final_dataset2.csv
[INFO] File size: 7897542 bytes
